In [17]:
import os
import sys
import glob
import logging
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

from typing import Literal
from pathlib import Path
from instanovo.utils.data_handler import SpectrumDataFrame

from instanovo.transformer.dataset import remove_modifications as clean_peptide

# Fix this later, imports should work without this
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

In [18]:
from common.utils import collect_files, get_or_create_folder
from common.logger import get_logger_config
from common.constants import (
    BASE_RAW_DATA_DIR,
    BASE_PROCESSED_DATA_DIR,
    BASE_LOGS_DIR,
    BASE_PLOTS_DIR,
    BASE_REPORTS_CSV_DIR,
)

In [19]:
logger_config = get_logger_config(subdir="scripts")
logging.config.dictConfig(logger_config)
logger = logging.getLogger(__name__)

In [20]:
# Collect each unique_peptide.csv file
peptides_file_paths = [
    path
    for path in collect_files(BASE_REPORTS_CSV_DIR, ext="csv")
    if "unique_peptides" in path
]

assert peptides_file_paths, peptides_file_paths

In [21]:
df = pd.concat([pd.read_csv(file) for file in peptides_file_paths], ignore_index=True)
df.head(20)

,Unique Peptides
0,HNGTGGR
1,SQNCHNSSSR
2,AAGMNHTK
3,ANASHDQPQK
4,HNDSGASECR
5,GGGGGGGGGGGGGSGSSSGSSTSR
6,RQQQQQQQQQQQQK
7,QQQQQQQQQQQQK
8,KNDSGAYR
9,KCLNHTTQK


In [22]:
df["Unique Peptides"].describe()

count                163595
unique                44976
top       AVCMLSNTTAIAEAWAR
freq                     10
Name: Unique Peptides, dtype: object

## Split without Kevin constraint

In [23]:
unique_peptides_df = df["Unique Peptides"].drop_duplicates()

In [24]:
indices = np.arange(len(unique_peptides_df))
np.random.shuffle(indices)
split_ratio = 0.8

split_seperator = int(len(unique_peptides_df) * split_ratio)

# Shuffle the DataFrame indices
shuffled_df = unique_peptides_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Train/test split
train_peptides_df = shuffled_df.iloc[:split_seperator].reset_index(drop=True)
test_peptides_df = shuffled_df.iloc[split_seperator:].reset_index(drop=True)

In [25]:
assert len(train_peptides_df) == 35980, len(train_peptides_df)

In [26]:
assert len(test_peptides_df) == 8996, len(test_peptides_df)

In [27]:
# The zero designs inherited by the SpectrumDataFrame class makes splitting the dataset
# one time complicated.Actually, when we filter an object of the SpectrumDataframe class, the filters
# are kept with the object and are lazily evaluated. So, when an object of the SpectrumDataframe class is filtered, a new object of the that class is not returned, but
# instead it is the old object that is mutated. So if, I decide to use the .filter_rows method, I'll have to filter train and test separately in different inner contexts. But I guess they should be a way to interact with the predicates held by an object of that class.

# v0 => for splitting algorithm without taking into account splitting suggestions from Kevin
# v1 =>


def write_split(
    project_name: Path | str,
    split_name: Literal["train", "val", "test"],  # noqa
    algorithm_version: Literal["vO", "v1", "v2"],
    potential_peptides_set: set,
    *args,
    max_charge: int = 10,
    **kwargs,
):
    logger.info(f"Instantiating SpectrumDataFrame with args={args} and kwargs={kwargs}")
    sdf = SpectrumDataFrame.load(*args, verbose=True, **kwargs)  # noqa
    logger.info(
        f"Instantiated SpectrumDataFrame with {len(sdf)} spectra from project {project_name}"
    )
    sdf.filter_rows(
        lambda row: (row["precursor_charge"] <= max_charge)
        and (row["precursor_charge"] > 0)
        and (row["peptide"] in potential_peptides_set)
    )
    logger.info(f"Got {len(sdf)} spectra after filtering by precursor charge")
    logger.info(f"Starting {split_name} split... for project {project_name}")
    target_path = BASE_PROCESSED_DATA_DIR / project_name
    sdf.save(target_path, partition=f"glyco_{algorithm_version}_{split_name}")
    logger.info(
        f"Saved {len(sdf)} spectra for {split_name} to {target_path} for project {project_name}"
    )
    return sdf

TypeError: unsupported operand type(s) for |: 'str' and 'str'

In [28]:
projects_dirs = glob.glob(f"{BASE_RAW_DATA_DIR}/*/")
assert projects_dirs, projects_dirs

In [ ]:
# Version 0 for train/test split
for project_dir in projects_dirs:
    project_name = project_dir.split("/")[-2]
    project_file_paths = collect_files(location=project_dir, ext="ipc")

    logger.info(
        f"Collected {len(project_file_paths)} of project {project_name} files from {project_dir}"
    )
    for split_name, peptide_set in [
        ("train", set(train_peptides_df)),
        # ("val", set(val_peptides_df)),
        ("test", set(test_peptides_df)),
    ]:

        write_split(
            split_name=split_name,
            algorithm_version="v0",
            potential_peptides_set=set(train_peptides_df),
            source=f"{BASE_RAW_DATA_DIR / project_name}/*",
            source_type="ipc",
            column_mapping={"intensity": "intensity_array", "mz": "mz_array"},
        )

2025-04-08 22:21:30,290 - __main__ - INFO - Collected 27 of project PXD026629 files from /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/
2025-04-08 22:21:30,314 - __main__ - INFO - Instantiating SpectrumDataFrame with args=() and kwargs={'source': '/home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/*', 'source_type': 'ipc', 'column_mapping': {'intensity': 'intensity_array', 'mz': 'mz_array'}}
2025-04-08 22:21:30,315 - instanovo.utils.data_handler - INFO - Loading file 001 of 027: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/20180904YLJ-VSV4h-02.ipc
2025-04-08 22:21:30,475 - instanovo.utils.data_handler - INFO - Loading file 002 of 027: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/20180904YLJ-VSV0h-03.ipc
2025-04-08 22:21:30,580 - instanovo.utils.data_handler - INFO - Loading file 003 of 027: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/

In [31]:
# sdf = write_split(
#     split_name="train",
#     algorithm_version="v0",
#     potential_peptides_set=set(train_peptides_df),
#     source=f"{BASE_RAW_DATA_DIR / 'PXD035158'}/*",
#     source_type="ipc",
#     column_mapping={"intensity": "intensity_array", "mz": "mz_array"},
# )

2025-04-08 00:24:56,356 - __main__ - INFO - Instantianting SpectrumDataFrame with args=() and kwargs={'source': '/home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD035158/*', 'source_type': 'ipc', 'column_mapping': {'intensity': 'intensity_array', 'mz': 'mz_array'}}
2025-04-08 00:24:56,361 - instanovo.utils.data_handler - INFO - Loading file 001 of 016: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD035158/Fut8_WT_max_IGP_mousebrain_2.mzML.ipc
2025-04-08 00:24:56,492 - instanovo.utils.data_handler - INFO - Loading file 002 of 016: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD035158/Fut8_WT_max_IGP_mousebrain_1.mzML.ipc
2025-04-08 00:24:56,648 - instanovo.utils.data_handler - INFO - Loading file 003 of 016: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD035158/LD_N39_serum_IGP_1.mzML.ipc
2025-04-08 00:24:56,724 - instanovo.utils.data_handler - INFO - Loading file 004 of 016: /home/hjisaac/A

ComputeError: TypeError: argument of type 'NoneType' is not iterable

35980

In [3]:
#
# rr  = pd.read_parquet(BASE_PROCESSED_DATA_DIR/ "PXD035158/dataset-ms-v0_train-0001-0001.parquet")
# rr.head()

,index,scan,header,rt,frag_type,collision_energy,precursor_mz,precursor_charge,precursor_intensity,lower_offset,...,peptide_observed_mz,peptide_calc_mz,delta_mass,expectation,hyperscore,nextscore,probability,auc_intensity,protein,experiment_name
0,7163,controllerType=0 controllerNumber=1 scan=7164,FTMS + c NSI d Full ms2 917.7194@hcd33.00 [120...,1035.362681,HCD,33.0,917.386108,3,9225750.000,1.0,...,917.3861,917.3799,0.0027,1.933290e-09,38.373,18.158,1.0000,96643616.0,sp|Q62443|NPTX1_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
1,7164,controllerType=0 controllerNumber=1 scan=7165,FTMS + c NSI d Full ms2 785.3360@hcd33.00 [120...,1035.628304,HCD,33.0,785.085449,4,8758446.000,1.0,...,785.0855,785.0808,0.0105,0.000000e+00,97.376,13.117,1.0000,74405520.0,sp|Q64012|RALY_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
2,7166,controllerType=0 controllerNumber=1 scan=7167,FTMS + c NSI d Full ms2 1020.4236@hcd33.00 [12...,1036.086781,HCD,33.0,1020.089478,3,4922692.500,1.0,...,1020.0895,1020.0836,-0.0010,6.758920e-03,31.962,18.467,0.7947,45616952.0,sp|Q62443|NPTX1_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
3,7170,controllerType=0 controllerNumber=1 scan=7171,FTMS + c NSI d Full ms2 973.4209@hcd33.00 [120...,1036.716772,HCD,33.0,973.087219,3,4103132.000,1.0,...,973.0872,973.0815,0.0001,7.019999e-03,27.510,17.685,0.9473,35005788.0,sp|Q60625|ICAM5_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
4,7174,controllerType=0 controllerNumber=1 scan=7175,FTMS + c NSI d Full ms2 803.0234@hcd33.00 [120...,1037.280757,HCD,33.0,802.689087,3,1636536.375,1.0,...,802.6891,802.6848,0.0080,2.104669e-02,25.936,16.479,0.8262,15937490.0,sp|Q60625|ICAM5_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
